# Figure 5 and Tables 11 and 12

Clue gains of two words rendered on one image, the clue grid of Table 11 and the renderings of Table 12.

Inputs: `results/two_clue_summary.parquet` from `scripts/analysis/figure5_two_clue.py`; the clue grids in `lexicon/two_clue_confirmatory.tsv`; the rendering conditions in `src/stimuli/two_clue_render.py`.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager as fm
from matplotlib.textpath import TextPath

from src.models import FAMILY_OF, PARAMS_M, PRETTY
from src.paper import style
from src.paper.results import sweep_rows, two_clue_summary
from src.stimuli.two_clue_items import read_table
from src.stimuli.two_clue_render import CONFIRMATORY_PATH, rendering_conditions

style.setup_notebook()
style.use_real_bold()

summary = two_clue_summary()
S = summary[summary.analysis_set.eq("confirmatory")].set_index("model")
PILOT = summary[summary.analysis_set.eq("pilot")]
ITEMS = read_table(CONFIRMATORY_PATH, "confirmatory")

## Table 11

The clue grid that crosses input and output with four kinds of signal, one target per cell.

In [ ]:
GRID = ITEMS[ITEMS.family_id.eq("direction_signal_confirmatory")]
table11 = (GRID.pivot(index="feature", columns="base", values="target")
           .reindex(index=GRID.feature.unique(), columns=GRID.base.unique())
           .rename_axis(index="Broad clue", columns="Narrow clue"))
display(table11)

shapes = ITEMS.groupby("family_id").agg(
    broad=("feature", "nunique"), narrow=("base", "nunique"),
    targets=("target", "nunique"), items=("target", "size")).drop_duplicates()
print(f"each of the {ITEMS.family_id.nunique()} clue grids: "
      + "; ".join(f"{row.broad} broad clues by {row.narrow} narrow clues, "
                  f"{row.targets} targets in {row.items} items" for row in shapes.itertuples()))

ITEM = GRID[GRID.feature.eq("input") & GRID.base.eq("sound")].iloc[0]
others = GRID[GRID.target.ne(ITEM.target)]
same_narrow = others.base.eq(ITEM.base)
same_broad = others.feature.eq(ITEM.feature)
print(f"target {ITEM.target}, from {ITEM.feature} with {ITEM.base}")
print(f"same narrow clue {ITEM.base}, other broad clue: {', '.join(others.target[same_narrow])}")
print(f"same broad clue {ITEM.feature}, other narrow clue: {', '.join(others.target[same_broad])}")
print(f"neither clue: {', '.join(others.target[~same_narrow & ~same_broad])}")
print(f"other targets the narrow clue {ITEM.base} alone separates from {ITEM.target}: "
      f"{int((~same_narrow).sum())} of {len(others)}")

## Table 12

The renderings of the item input with sound, and the target rendered alone with its control image.

In [ ]:
RENDERING = {"both_readable": "Both intact", "feature_readable": "Broad intact",
             "base_readable": "Narrow intact", "both_scrambled": "Neither intact",
             "target_readable": "Target alone", "target_scrambled": "Target control"}
DIFFERENCE = {"both_readable": "τ̂_both", "feature_readable": "τ̂_broad",
              "base_readable": "τ̂_narrow",
              "both_scrambled": "control subtracted from τ̂_both, τ̂_broad and τ̂_narrow",
              "target_readable": "τ̂_target",
              "target_scrambled": "control subtracted from τ̂_target"}

rows = []
for condition, drawn in rendering_conditions(ITEM).items():
    shown = [word if mode == "readable" else "phase-randomized" for word, mode in drawn]
    clues = shown if len(shown) == 2 else ["", ""]
    target = shown[0] if len(shown) == 1 else ""
    rows.append({"Rendering": RENDERING[condition], "Broad clue": clues[0],
                 "Narrow clue": clues[1], "Target": target,
                 "Difference vector": DIFFERENCE[condition]})
table12 = pd.DataFrame(rows).set_index("Rendering")
table12

## Clue gains and corrected tests

Gains, 95 percent bootstrap intervals and Holm-corrected p for each model on the twelve confirmatory clue grids.

In [ ]:
BROAD = "feature_axis_gain_over_base"
NARROW = "base_axis_gain_over_feature"
RESIDUAL = "interaction_margin"

gains = sweep_rows(S.reset_index()).set_index("label")[[
    BROAD, f"{BROAD}_ci_lo", f"{BROAD}_ci_hi",
    NARROW, f"{NARROW}_ci_lo", f"{NARROW}_ci_hi",
    "structured_iut_p_holm", "joint_margin", "joint_margin_p_holm",
    RESIDUAL, f"{RESIDUAL}_ci_lo", f"{RESIDUAL}_ci_hi", f"{RESIDUAL}_p_holm"]]
gains.columns = ["broad-clue gain", "broad lo", "broad hi",
                 "narrow-clue gain", "narrow lo", "narrow hi", "smaller gain Holm p",
                 "margin over seven", "margin Holm p",
                 "residual margin", "residual lo", "residual hi", "residual Holm p"]
display(gains)

n_grids = int(S.n_families.iloc[0])
print(f"confirmatory set: {n_grids} clue grids, {int(S.n_equations.iloc[0])} items")
print(f"pilot set, scored separately: {int(PILOT.n_families.iloc[0])} clue grids")
print(f"Holm correction across {int(S.structured_iut_p_holm.notna().sum())} "
      f"pretrained models")

smaller = S.structured_iut_p_holm.nsmallest(2)
print("smaller gain, Holm p: "
      + ", then ".join(f"{PRETTY[m]} {p:.4f}" for m, p in smaller.items()))


def interval(model, column):
    return f"[{S.loc[model, f'{column}_ci_lo']:.4f}, {S.loc[model, f'{column}_ci_hi']:.4f}]"


for column, axis in ((BROAD, "broad"), (NARROW, "narrow")):
    excludes = S.index[S[f"{column}_ci_lo"].gt(0) | S[f"{column}_ci_hi"].lt(0)]
    print(f"{axis}-clue gain interval excludes zero: "
          f"{', '.join(PRETTY[m] for m in excludes)}")
for model in ("siglip2-b", "dinov3-7b", "dinov3-h", "vit-random-b"):
    print(f"{PRETTY[model]}: broad {interval(model, BROAD)}, "
          f"narrow {interval(model, NARROW)}")

residual = S[f"{RESIDUAL}_p_holm"]
print(f"residual margin, smallest Holm p: {PRETTY[residual.idxmin()]} {residual.min():.3f}")
for model in S.index[S[f"{RESIDUAL}_ci_hi"].lt(0)]:
    print(f"{PRETTY[model]} residual margin {S.loc[model, RESIDUAL]:.6f}, interval "
          f"[{S.loc[model, f'{RESIDUAL}_ci_lo']:.6f}, {S.loc[model, f'{RESIDUAL}_ci_hi']:.6f}]")

# Models whose margin over all seven other targets has a Holm p below this are drawn in color.
SIGNAL_HOLM = 0.10
SIGNAL = tuple(S.index[S.joint_margin_p_holm < SIGNAL_HOLM])
print(f"margin over all seven other targets, Holm p below {SIGNAL_HOLM:.2f}: "
      + ", ".join(f"{PRETTY[m]} {S.loc[m, 'joint_margin_p_holm']:.4f}" for m in SIGNAL))

print(f"95 percent percentile intervals from {int(S.n_bootstrap.iloc[0]):,} "
      f"bootstrap resamples of the {n_grids} clue grids")

## Figure 5

Broad-clue gain (a) and narrow-clue gain (b) with their bootstrap intervals, one row per model.

In [ ]:
# `(column, name, subtitle)` of each panel.
PANELS = (
    (BROAD, "Broad-clue gain", "Both clues over the narrow clue"),
    (NARROW, "Narrow-clue gain", "Both clues over the broad clue"),
)
FLOOR = "vit-random-b"

HEIGHT_PT = 152.0
GUTTER_PT = 31.0
PANEL_PT = 100.0

INTERVAL_LW = 1.0
# Diameter of the estimate mark, in points.
MARK_PT = 2.6
MARK = np.pi * (MARK_PT / 2.0) ** 2
RAD = MARK_PT / 2.0

# Width a panel receives, in points. The gutter and the two panels divide the
# page width, less the padding, by their width ratios, so a panel gets 100 of
# 231 parts.
PANEL_DELIVERED_PT = 163.0

# Clearance from the panel edge, in points, for a mark, an interval end and zero.
MARK_ROOM_PT = RAD + 0.9
TIP_ROOM_PT = 5.0
ZERO_ROOM_PT = 4.0

# Tick steps the axis may take, and the tick counts it accepts, preferred first.
TICK_STEPS = tuple(s * 10.0 ** e for e in range(-4, 1)
                   for s in (1.0, 2.0, 2.5, 4.0, 5.0))
TICK_COUNTS = (3, 4)

HEADER_LETTER_GAP_PT = 3.2
HEADER_BASE_Y = 1.0
HEADER_NAME_RISE_PT = 10.0
HEADER_SUBTITLE_RISE_PT = 3.0
HEADER_BAND_PT = 22.0

FAMILY_ORDER = ("DINOv3", "DINOv2", "MAE", "Web-DINO", "CLIP", "SigLIP2",
                "Untrained")


def bands() -> tuple[tuple[str, tuple[str, ...]], ...]:
    """Models by family, largest first within a family."""
    return tuple(
        (family, tuple(sorted((m for m in S.index if FAMILY_OF[m] == family),
                              key=lambda m: -PARAMS_M[m])))
        for family in FAMILY_ORDER)


BANDS = bands()
SHADED = set(FAMILY_ORDER[::2])


def row_plan() -> tuple[list[tuple], list[tuple]]:
    """Each row's vertical position, model and color, and each family's vertical span."""
    plan, spans, y = [], [], 0.0
    for family, models in BANDS:
        top = y + 0.5
        for m in models:
            plan.append((y, m, style.REFERENCE if m == FLOOR
                         else style.SEMANTIC if m in SIGNAL else style.CONTROL))
            y -= 1.0
        spans.append((family, top, y + 0.5))
    return plan, spans


ROWS, SPANS = row_plan()
YLIM = (min(y for y, *_ in ROWS) - 0.5, 0.5)


def anchors() -> list[tuple[float, float]]:
    """Every value the figure draws, with the clearance it needs."""
    out = [(0.0, ZERO_ROOM_PT)]
    for column, _, _ in PANELS:
        out += [(float(v), MARK_ROOM_PT) for v in S[column]]
        out += [(float(v), TIP_ROOM_PT) for v in S[f"{column}_ci_lo"]]
        out += [(float(v), TIP_ROOM_PT) for v in S[f"{column}_ci_hi"]]
    return out


def fitted_limits(anchor_list, panel_pt=PANEL_DELIVERED_PT):
    """Axis limits that hold every anchor with its clearance, and their ticks.

    The clearance is in points and the limits are in data units, so the span is
    iterated to its fixed point.
    """
    span = (max(v for v, _ in anchor_list) - min(v for v, _ in anchor_list))
    lo = hi = 0.0
    for _ in range(64):
        lo = min(v - room * span / panel_pt for v, room in anchor_list)
        hi = max(v + room * span / panel_pt for v, room in anchor_list)
        if abs((hi - lo) - span) < 1e-15:
            break
        span = hi - lo
    for wanted in TICK_COUNTS:
        for step in sorted(set(TICK_STEPS), reverse=True):
            ticks = [round(i * step, 10) for i in
                     range(math.ceil(lo / step), math.floor(hi / step) + 1)]
            if len(ticks) == wanted:
                return (lo, hi), tuple(ticks)
    raise RuntimeError(f"no step puts three or four ticks inside "
                       f"[{lo:.4f}, {hi:.4f}]")


XLIM, XTICKS = fitted_limits(anchors())


def tick_label(value: float) -> str:
    """A tick as the paper prints a proportion, without its leading zero."""
    text = f"{value:.4f}".rstrip("0").rstrip(".")
    if text in ("0", "1", "-0"):
        return "0" if text == "-0" else text
    return text.replace("0.", ".", 1)


def ink_width(text: str, size: float, weight: str = "normal") -> float:
    """The right edge of the ink of `text` at `size`, from its origin."""
    prop = fm.FontProperties(family=plt.rcParams["font.sans-serif"][0],
                             weight=weight)
    return float(TextPath((0, 0), text, size=size, prop=prop).get_extents().x1)


# One leader per row, under everything.
LEADER = dict(color=style.GRID, lw=style.LEADER_WIDTH,
              ls=style.DASH["construction"], zorder=0)
# Zero, at the weight of the bottom spine.
ZERO_RULE = dict(color=style.CHANCE, ls=style.DASH["chance"], lw=0.45, zorder=1)


def draw_bands(axes):
    """Tint each shaded family inside every panel, so the gap between panels breaks the band."""
    for ax in axes[1:]:
        for family, top, bottom in SPANS:
            if family not in SHADED:
                continue
            hi, lo = min(top, YLIM[1]), max(bottom, YLIM[0])
            ax.axhspan(lo, hi, facecolor=style.FAINT, edgecolor="none",
                       zorder=-1)


def draw_header(ax, letter, name, subtitle):
    """The panel letter and name, over the subtitle, kept out of the layout."""
    indent = ink_width(letter, style.PANEL_SIZE, "bold") + HEADER_LETTER_GAP_PT
    for dx, rise, text, size, colour, weight in (
            (0.0, HEADER_NAME_RISE_PT, letter, style.PANEL_SIZE, style.INK, "bold"),
            (indent, HEADER_NAME_RISE_PT, name, style.PANEL_SIZE, style.INK, "bold"),
            (0.0, HEADER_SUBTITLE_RISE_PT, subtitle, style.SUBTITLE_SIZE,
             style.CONTROL, "normal")):
        t = ax.annotate(text, (0.0, HEADER_BASE_Y), xycoords="axes fraction",
                        xytext=(dx, rise), textcoords="offset points",
                        ha="left", va="baseline", fontsize=size, color=colour,
                        fontweight=weight, annotation_clip=False)
        t.set_in_layout(False)


def draw_rows(ax, column):
    """The leader, interval and estimate of every row of one panel."""
    for y, model, colour in ROWS:
        ax.plot([0, 1], [y, y], transform=ax.get_yaxis_transform(), **LEADER)
        lo = float(S.loc[model, f"{column}_ci_lo"])
        hi = float(S.loc[model, f"{column}_ci_hi"])
        est = float(S.loc[model, column])
        ax.plot([lo, hi], [y, y], color=colour, lw=INTERVAL_LW,
                solid_capstyle="butt", zorder=2)
        ax.scatter(est, y, s=MARK, marker="o", facecolor=colour,
                   edgecolor="none", linewidth=0.0, zorder=3)


def draw_gutter(gax):
    """The model names, right-aligned against the first panel."""
    gax.set_axis_off()
    gax.set_xlim(0, 1)
    gax.set_ylim(*YLIM)
    for y, model, _ in ROWS:
        gax.text(1.0, y, PRETTY[model], transform=gax.get_yaxis_transform(),
                 ha="right", va="center", fontsize=style.TEXT_SIZE,
                 color=style.CONTROL)


def style_panel(ax):
    ax.axvline(0.0, **ZERO_RULE)
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.set_yticks([])
    ax.set_xticks(list(XTICKS), [tick_label(t) for t in XTICKS])
    ax.tick_params(axis="x", labelsize=style.TEXT_SIZE, length=2, pad=1.5)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_linewidth(0.45)


def draw_figure():
    fig, axes = plt.subplots(
        1, 1 + len(PANELS), figsize=style.inches(HEIGHT_PT, style.TEXT_WIDTH_IN),
        layout="constrained",
        gridspec_kw=dict(width_ratios=(GUTTER_PT,) + (PANEL_PT,) * len(PANELS)))
    fig.get_layout_engine().set(w_pad=0.012, h_pad=0.010, wspace=0.045,
                                rect=(0, 0, 1, 1 - HEADER_BAND_PT / HEIGHT_PT))
    draw_gutter(axes[0])
    for i, (ax, (column, name, subtitle)) in enumerate(zip(axes[1:], PANELS)):
        draw_header(ax, f"({chr(97 + i)})", name, subtitle)
        style_panel(ax)
        draw_rows(ax, column)
    draw_bands(axes)
    return fig

In [ ]:
fig = draw_figure()
# Constrained layout settles on the third pass for this figure, and style.save runs two.
fig.canvas.draw()
style.save(fig, "figure5_two_clue")